# Limpeza e Preparação de Dados - Passos Mágicos Datathon

## Objetivo
Padronizar, limpar e unificar os dados de três anos (2022-2024) para uma análise abrangente.

## Processo
1. Padronizar nomes de colunas entre os anos
2. Remover colunas com 100% de ausência (artefatos de metadata)
3. Consolidar colunas dos indicadores principais
4. Criar um dataset unificado multi-ano
5. Validar e documentar as transformações

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Setup
project_root = Path.cwd().parent
DATA_FILE = project_root / "BASE DE DADOS PEDE 2024 - DATATHON.xlsx"
DATA_DIR = project_root / "data"
DATA_DIR.mkdir(exist_ok=True)

print(f"Working directory: {project_root}")
print(f"Data directory: {DATA_DIR}")

Working directory: c:\Users\cso2569\Python\Pos-Tech-Data-Analytics\Modulo 5\fiap-datathon-fase5
Data directory: c:\Users\cso2569\Python\Pos-Tech-Data-Analytics\Modulo 5\fiap-datathon-fase5\data


## 1. Carregar Dados Brutos

In [ ]:
# Carregar todos os sheets
data_dict = {}
excel_file = pd.ExcelFile(DATA_FILE)

for sheet in excel_file.sheet_names:
    df = pd.read_excel(excel_file, sheet_name=sheet)
    data_dict[sheet] = df
    print(f"{sheet}: {df.shape[0]} linhas × {df.shape[1]} colunas")

PEDE2022: 860 rows × 42 columns
PEDE2023: 1014 rows × 48 columns
PEDE2024: 1156 rows × 50 columns


## 2. Definir Mapeamento de Colunas

Mapear variações de nomes de colunas para nomes canônicos.

In [ ]:
# Colunas canônicas - indicadores principais presentes em todos os anos
CANONICAL_COLUMNS = {
    # Identifiers
    'RA': 'student_id',
    'Fase': 'phase',
    'Turma': 'class',
    'Nome': 'name',
    
    # Demographics
    'Ano nasc': 'birth_year',
    'Idade 22': 'age_2022',
    'Idade': 'age',
    'Gênero': 'gender',
    'Ano ingresso': 'admission_year',
    'Instituição de ensino': 'school_institution',
    'Escola': 'school_institution',
    
    # Core Indicators (main analysis focus)
    'IAN': 'ian',  # Academic adequacy
    'IDA': 'ida',  # Academic performance
    'IEG': 'ieg',  # Engagement
    'IAA': 'iaa',  # Self-assessment
    'IPS': 'ips',  # Psychosocial aspects
    'IPP': 'ipp',  # Psychopedagogical
    'IPV': 'ipv',  # Turning point
    'INDE 22': 'inde_2022',
    'INDE 23': 'inde_2023',
    'INDE 2023': 'inde_2023',
    'INDE 2024': 'inde_2024',
    'INDE 24': 'inde_2024',
    
    # Academic Subjects
    'Matem': 'math',
    'Mat': 'math',
    'Portug': 'portuguese',
    'Por': 'portuguese',
    'Inglês': 'english',
    'Ing': 'english',
    
    # Deficiency indicator
    'Defas': 'deficiency',
    'IAN': 'ian',
    
    # Turning point
    'Atingiu PV': 'achieved_turning_point',
    'Indicado': 'indicated_for_intervention',
}

# Colunas a excluir (100% faltantes ou artefatos de metadata)
COLUMNS_TO_EXCLUDE = {
    'Destaque IPV.1', 'Destaque IPV', 'Destaque IEG', 'Destaque IDA',  # Duplicates/artifacts
    'Rec Av1', 'Rec Av2', 'Rec Av3', 'Rec Av4',  # 100% missing recommendation columns
    'Rec Psicologia',  # 100% missing
    'Avaliador5', 'Avaliador6',  # 87-99% missing, doesn't exist in early years
    'Cg', 'Cf', 'Ct',  # Internal scoring system not relevant for analysis
    'Nº Av',  # Number of evaluators - metadata
    'Avaliador1', 'Avaliador2', 'Avaliador3', 'Avaliador4',  # Evaluator names not needed
    'Pedra 20', 'Pedra 21', 'Pedra 22', 'Pedra 23', 'Pedra 2024',  # Stage markers, not analysis targets
    'Pedra 23',
}

print(f"Colunas canônicas definidas: {len(CANONICAL_COLUMNS)}")
print(f"Colunas a excluir: {len(COLUMNS_TO_EXCLUDE)}")

Canonical columns defined: 32
Columns to exclude: 24


## 3. Limpar Dados por Ano

In [4]:
def clean_year_data(df, year_label):
    """
    Clean single year's data:
    1. Rename columns
    2. Remove excluded columns
    3. Add year identifier
    4. Keep only analyzable columns
    """
    df_clean = df.copy()
    
    # Remove duplicate columns first (keep first occurrence)
    df_clean = df_clean.loc[:, ~df_clean.columns.duplicated(keep='first')]
    
    # Filter to columns we want to rename
    cols_to_rename = {col: CANONICAL_COLUMNS[col] for col in df_clean.columns if col in CANONICAL_COLUMNS}
    df_clean.rename(columns=cols_to_rename, inplace=True)
    
    # Remove excluded columns
    cols_to_drop = [col for col in df_clean.columns if col in COLUMNS_TO_EXCLUDE]
    df_clean.drop(columns=cols_to_drop, inplace=True, errors='ignore')
    
    # Remove columns with 100% missing (except for expected ones)
    missing_pct = (df_clean.isnull().sum() / len(df_clean) * 100)
    cols_100_missing = missing_pct[missing_pct == 100.0].index.tolist()
    # Don't drop core indicators even if missing (might be populated in other years)
    core_indicators = ['ian', 'ida', 'ieg', 'iaa', 'ips', 'ipp', 'ipv']
    cols_100_missing = [col for col in cols_100_missing if col not in core_indicators]
    df_clean.drop(columns=cols_100_missing, inplace=True, errors='ignore')
    
    # Remove duplicate columns that might have appeared after renaming
    df_clean = df_clean.loc[:, ~df_clean.columns.duplicated(keep='first')]
    
    # Add year identifier
    df_clean['year'] = year_label
    
    return df_clean

# Clean each year
data_cleaned = {}
for year_label, df in data_dict.items():
    df_clean = clean_year_data(df, year_label)
    data_cleaned[year_label] = df_clean
    print(f"\n{year_label}:")
    print(f"  Original: {df.shape[1]} columns")
    print(f"  Cleaned: {df_clean.shape[1]} columns")
    print(f"  Remaining columns: {df_clean.shape[0]} rows")
    # Check for duplicate columns
    if df_clean.columns.duplicated().any():
        print(f"  WARNING: Duplicate columns detected!")


PEDE2022:
  Original: 42 columns
  Cleaned: 24 columns
  Remaining columns: 860 rows

PEDE2023:
  Original: 48 columns
  Cleaned: 24 columns
  Remaining columns: 1014 rows

PEDE2024:
  Original: 50 columns
  Cleaned: 27 columns
  Remaining columns: 1156 rows


## 4. Padronizar e Mesclar Anos

In [ ]:
# Obter todas as colunas únicas nos datasets limpos
all_columns = set()
for df in data_cleaned.values():
    all_columns.update(df.columns)

print(f"Todas as colunas únicas entre os anos: {len(all_columns)}")
print(f"Colunas: {sorted(all_columns)}")

All unique columns across years: 35
Columns: ['Ativo/ Inativo', 'Ativo/ Inativo.1', 'Data de Nasc', 'Defasagem', 'Fase Ideal', 'Fase ideal', 'Nome Anonimizado', 'Pedra 2023', 'achieved_turning_point', 'admission_year', 'age', 'age_2022', 'birth_year', 'class', 'deficiency', 'english', 'gender', 'iaa', 'ian', 'ida', 'ieg', 'inde_2022', 'inde_2023', 'inde_2024', 'indicated_for_intervention', 'ipp', 'ips', 'ipv', 'math', 'name', 'phase', 'portuguese', 'school_institution', 'student_id', 'year']


In [ ]:
# Remover colunas duplicadas e criar conjunto padronizado
for year_label in data_cleaned.keys():
    df = data_cleaned[year_label]
    # Manter apenas colunas sem sufixos .1, .2 (duplicatas)
    df_clean = df.loc[:, ~df.columns.str.contains(r'\.\d+$', regex=True)]
    data_cleaned[year_label] = df_clean

# Obter colunas únicas limpas
all_columns_clean = set()
for df in data_cleaned.values():
    all_columns_clean.update(df.columns)

print(f"Colunas únicas limpas: {len(all_columns_clean)}")
print(f"Colunas: {sorted(all_columns_clean)}")

# Garantir que todos os dataframes tenham as mesmas colunas
for year_label in data_cleaned.keys():
    for col in all_columns_clean:
        if col not in data_cleaned[year_label].columns:
            data_cleaned[year_label][col] = np.nan

# Mesclar todos os anos
dfs_to_concat = []
for year_label in ['PEDE2022', 'PEDE2023', 'PEDE2024']:
    df_temp = data_cleaned[year_label].copy()
    df_temp = df_temp.reset_index(drop=True)
    dfs_to_concat.append(df_temp)

df_unified = pd.concat(dfs_to_concat, ignore_index=True, sort=False)

print(f"\nDataset unificado: {df_unified.shape[0]} linhas × {df_unified.shape[1]} colunas")
print(f"Anos representados: {df_unified['year'].unique()}")
print(f"\nLinhas por ano:")
print(df_unified['year'].value_counts().sort_index())

Clean unique columns: 34
Columns: ['Ativo/ Inativo', 'Data de Nasc', 'Defasagem', 'Fase Ideal', 'Fase ideal', 'Nome Anonimizado', 'Pedra 2023', 'achieved_turning_point', 'admission_year', 'age', 'age_2022', 'birth_year', 'class', 'deficiency', 'english', 'gender', 'iaa', 'ian', 'ida', 'ieg', 'inde_2022', 'inde_2023', 'inde_2024', 'indicated_for_intervention', 'ipp', 'ips', 'ipv', 'math', 'name', 'phase', 'portuguese', 'school_institution', 'student_id', 'year']

Unified dataset: 3030 rows × 34 columns
Years represented: ['PEDE2022' 'PEDE2023' 'PEDE2024']

Rows per year:
year
PEDE2022     860
PEDE2023    1014
PEDE2024    1156
Name: count, dtype: int64


## 5. Validar Indicadores Principais

In [ ]:
# Verificar disponibilidade dos indicadores principais
core_indicators = ['ian', 'ida', 'ieg', 'iaa', 'ips', 'ipp', 'ipv']

print("Disponibilidade dos Indicadores Principais:")
print("="*80)

for indicator in core_indicators:
    if indicator in df_unified.columns:
        missing_pct = (df_unified[indicator].isnull().sum() / len(df_unified) * 100)
        stats = df_unified[indicator].describe()
        print(f"\n{indicator.upper()}:")
        print(f"  Faltantes: {df_unified[indicator].isnull().sum()} ({missing_pct:.1f}%)")
        print(f"  Intervalo: {stats['min']:.1f} - {stats['max']:.1f}")
        print(f"  Média: {stats['mean']:.2f}, Desvio Padrão: {stats['std']:.2f}")
    else:
        print(f"\n{indicator.upper()}: NÃO ENCONTRADO")

Core Indicators Availability:

IAN:
  Missing: 0 (0.0%)
  Range: 2.5 - 10.0
  Mean: 7.18, Std: 2.54

IDA:
  Missing: 178 (5.9%)
  Range: 0.0 - 10.0
  Mean: 6.38, Std: 1.96

IEG:
  Missing: 76 (2.5%)
  Range: 0.0 - 10.0
  Mean: 7.95, Std: 2.15

IAA:
  Missing: 165 (5.4%)
  Range: 0.0 - 10.0
  Mean: 7.92, Std: 2.63

IPS:
  Missing: 171 (5.6%)
  Range: 2.5 - 10.0
  Mean: 6.29, Std: 1.79

IPP:
  Missing: 1038 (34.3%)
  Range: 2.5 - 10.0
  Mean: 7.56, Std: 0.94

IPV:
  Missing: 178 (5.9%)
  Range: 2.5 - 10.0
  Mean: 7.55, Std: 1.08


## 6. Tratar Valores Faltantes nos Indicadores Principais

Para os indicadores principais, apenas documentar os valores faltantes e manter os registros por enquanto.

In [ ]:
# Linhas com indicadores principais faltantes por ano
print("Linhas com Indicadores Principais Faltantes por Ano:")
print("="*80)

for year in ['PEDE2022', 'PEDE2023', 'PEDE2024']:
    df_year = df_unified[df_unified['year'] == year]
    print(f"\n{year}:")
    
    for indicator in core_indicators:
        if indicator in df_year.columns:
            missing = df_year[indicator].isnull().sum()
            missing_pct = (missing / len(df_year) * 100)
            print(f"  {indicator}: {missing} ({missing_pct:.1f}%)")

Rows with Missing Core Indicators by Year:

PEDE2022:
  ian: 0 (0.0%)
  ida: 0 (0.0%)
  ieg: 0 (0.0%)
  iaa: 0 (0.0%)
  ips: 0 (0.0%)
  ipp: 860 (100.0%)
  ipv: 0 (0.0%)

PEDE2023:
  ian: 0 (0.0%)
  ida: 77 (7.6%)
  ieg: 76 (7.5%)
  iaa: 63 (6.2%)
  ips: 69 (6.8%)
  ipp: 76 (7.5%)
  ipv: 76 (7.5%)

PEDE2024:
  ian: 0 (0.0%)
  ida: 101 (8.7%)
  ieg: 0 (0.0%)
  iaa: 102 (8.8%)
  ips: 102 (8.8%)
  ipp: 102 (8.8%)
  ipv: 102 (8.8%)


## 7. Salvar Datasets Limpos

In [ ]:
# Salvar dataset unificado
output_path_unified = DATA_DIR / "dados_unificados.csv"
df_unified.to_csv(output_path_unified, index=False, encoding='utf-8')
print(f"Dataset unificado salvo: {output_path_unified}")

# Salvar anos individuais (para referência)
for year_label, df_year_clean in data_cleaned.items():
    output_path = DATA_DIR / f"dados_{year_label.lower()}.csv"
    df_year_clean.to_csv(output_path, index=False, encoding='utf-8')
    print(f"Salvo: {output_path}")

print(f"\nLimpeza de dados concluída!")
print(f"\nDataset pronto para análise:")
print(f"  Total de registros: {len(df_unified)}")
print(f"  Total de colunas: {df_unified.shape[1]}")
print(f"  Indicadores principais presentes: {len([c for c in core_indicators if c in df_unified.columns])}")

Unified dataset saved: c:\Users\cso2569\Python\Pos-Tech-Data-Analytics\Modulo 5\fiap-datathon-fase5\data\dados_unificados.csv
Saved: c:\Users\cso2569\Python\Pos-Tech-Data-Analytics\Modulo 5\fiap-datathon-fase5\data\dados_pede2022.csv
Saved: c:\Users\cso2569\Python\Pos-Tech-Data-Analytics\Modulo 5\fiap-datathon-fase5\data\dados_pede2023.csv
Saved: c:\Users\cso2569\Python\Pos-Tech-Data-Analytics\Modulo 5\fiap-datathon-fase5\data\dados_pede2024.csv

Data cleaning complete!

Dataset ready for analysis:
  Total records: 3030
  Total columns: 34
  Core indicators present: 7


## 8. Resumo

Preparação de dados concluída. O dataset unificado está pronto para as perguntas analíticas e para a modelagem.